# Apache Polaris — Data Layer Verification

**Date:** 2026-05-15  
**Goal:** Verify the three-layer architecture of Polaris

```
PostgreSQL (Polaris Metastore)
    ↓ points to
MinIO/metadata (Iceberg spec)
    ↓ points to
MinIO/data (Parquet files)
```

In [11]:
# ============================================================
# Prerequisites
# ============================================================
# Run this cell first before any other cells.
#
# If using uv (recommended):
#   uv add requests pandas pyvis boto3
#
# If using pip:
#   pip install requests pandas pyvis boto3
# ============================================================

import subprocess
import sys
import os

REQUIRED = ['requests', 'pandas', 'pyvis', 'boto3']

# Try uv first, fall back to pip
def install(pkg):
    # Try uv
    result = subprocess.run(
        ['uv', 'add', pkg],
        capture_output=True, text=True,
        cwd=os.getcwd()
    )
    if result.returncode == 0:
        return '✅', 'uv'
    
    # Fall back to pip
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        return '✅', 'pip'
    
    return '❌', 'failed'

print('Installing dependencies...')
for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')

print('\n⚠️  Restart kernel after first install.')
print('   Kernel menu → Restart Kernel → Run All Cells Below')

# After install, import immediately
import importlib
import requests
import pandas as pd

print('✅ Imports successful')

Installing dependencies...
  ✅ requests (uv)
  ✅ pandas (uv)
  ✅ pyvis (uv)
  ✅ boto3 (uv)

⚠️  Restart kernel after first install.
   Kernel menu → Restart Kernel → Run All Cells Below
✅ Imports successful


In [12]:
# Cell 1 — Setup
import requests
import json
import pandas as pd
from IPython.display import display, JSON

POLARIS_URL = 'http://192.168.139.2:8181'
MINIO_URL   = 'http://192.168.139.2:9000'
REALM       = 'POLARIS'
MINIO_USER  = 'admin'
MINIO_PASS  = 'bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0'
BUCKET      = 'data-catalog-bucket'

# Target entities
CATALOG   = 'test-catalog'
NAMESPACE = 'test_namespace'
TABLE     = 'test_table'

def get_token(client_id='root', client_secret='polaris-secret'):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

def polaris_headers(token):
    return {
        'Authorization': f'Bearer {token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json'
    }

TOKEN = get_token()
print(f'✅ Connected to Polaris')
print(f'   Target: {CATALOG}.{NAMESPACE}.{TABLE}')

✅ Connected to Polaris
   Target: test-catalog.test_namespace.test_table


## Layer 1 — PostgreSQL (Polaris Metastore)

Polaris stores all entity metadata in PostgreSQL.
We verify this via the Management API — which reads directly from PostgreSQL.

In [4]:
# Cell 2 — Layer 1: Verify entities in PostgreSQL via Polaris API
print('=' * 60)
print('LAYER 1: PostgreSQL (Polaris Metastore)')
print('=' * 60)

# Principals
principals = requests.get(
    f'{POLARIS_URL}/api/management/v1/principals',
    headers=polaris_headers(TOKEN)
).json().get('principals', [])

print(f'\n📋 Principals ({len(principals)})')
df = pd.DataFrame(principals)[['name', 'clientId', 'entityVersion', 'createTimestamp', 'lastUpdateTimestamp']]
display(df)

# Catalogs
catalogs = requests.get(
    f'{POLARIS_URL}/api/management/v1/catalogs',
    headers=polaris_headers(TOKEN)
).json().get('catalogs', [])

print(f'\n📚 Catalogs ({len(catalogs)})')
for c in catalogs:
    location = c.get('properties', {}).get('default-base-location', 'N/A')
    storage  = c.get('storageConfigInfo', {}).get('storageType', 'N/A')
    print(f'  Name          : {c["name"]}')
    print(f'  Type          : {c.get("type", "N/A")}')
    print(f'  Storage       : {storage}')
    print(f'  Location      : {location}')
    print(f'  entityVersion : {c.get("entityVersion", "N/A")}')
    print()

# Namespaces
print(f'\n📁 Namespaces')
for c in catalogs:
    cname = c['name']
    ns = requests.get(
        f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces',
        headers=polaris_headers(TOKEN)
    ).json().get('namespaces', [])
    for n in ns:
        print(f'  {cname} → {n}')

# Tables
print(f'\n📊 Tables')
for c in catalogs:
    cname = c['name']
    ns_list = requests.get(
        f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces',
        headers=polaris_headers(TOKEN)
    ).json().get('namespaces', [])
    for ns in ns_list:
        ns_name = ns[0] if isinstance(ns, list) else ns
        tables = requests.get(
            f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables',
            headers=polaris_headers(TOKEN)
        ).json().get('identifiers', [])
        for t in tables:
            tname = t.get('name', str(t))
            print(f'  {cname}.{ns_name}.{tname}')

print('\n✅ Layer 1 verified — all entities stored in PostgreSQL')

LAYER 1: PostgreSQL (Polaris Metastore)

📋 Principals (2)


,name,clientId,entityVersion,createTimestamp,lastUpdateTimestamp
0,root,root,1,1778724023496,1778724023496
1,spark-service,61f86631c73fcae9,1,1778731831759,1778731831759



📚 Catalogs (1)
  Name          : test-catalog
  Type          : INTERNAL
  Storage       : S3
  Location      : s3://data-catalog-bucket/test-catalog
  entityVersion : 1


📁 Namespaces
  test-catalog → ['test_namespace']

📊 Tables
  test-catalog.test_namespace.test_table

✅ Layer 1 verified — all entities stored in PostgreSQL


## Layer 2 — MinIO/metadata (Iceberg spec)

Polaris returns the exact Iceberg metadata location.
We fetch the actual `metadata.json` from MinIO to verify the Iceberg spec.

In [7]:
# Cell 3 — Layer 2: Verify Iceberg metadata in MinIO
import boto3
from botocore.client import Config

# MinIO client with AWS Signature V4
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_URL,
    aws_access_key_id=MINIO_USER,
    aws_secret_access_key=MINIO_PASS,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Get metadata location from Polaris
table_meta = requests.get(
    f'{POLARIS_URL}/api/catalog/v1/{CATALOG}/namespaces/{NAMESPACE}/tables/{TABLE}',
    headers=polaris_headers(TOKEN)
).json()

metadata_location = table_meta.get('metadata-location', '')
print(f'📍 Metadata location: {metadata_location}')

# Parse path
s3_path = metadata_location.replace('s3a://', '').replace('s3://', '')
bucket  = s3_path.split('/')[0]
key     = '/'.join(s3_path.split('/')[1:])

# Fetch metadata.json from MinIO
import json
obj = s3.get_object(Bucket=bucket, Key=key)
iceberg_meta = json.loads(obj['Body'].read())

print(f'\n✅ Iceberg metadata.json fetched')
print(f'  format-version : {iceberg_meta.get("format-version")}')
print(f'  table-uuid     : {iceberg_meta.get("table-uuid")}')
print(f'  location       : {iceberg_meta.get("location")}')

print(f'\n📐 Schema:')
schema = iceberg_meta.get('schemas', [{}])[0]
for field in schema.get('fields', []):
    print(f'  [{field["id"]}] {field["name"]} : {field["type"]}')

print(f'\n📸 Snapshots:')
for snap in iceberg_meta.get('snapshots', []):
    print(f'  snapshot-id   : {snap.get("snapshot-id")}')
    print(f'  operation     : {snap.get("summary", {}).get("operation")}')
    print(f'  added-records : {snap.get("summary", {}).get("added-records")}')

📍 Metadata location: s3a://data-catalog-bucket/test-catalog/test_namespace/test_table/metadata/00001-a63c7286-ab57-4206-847a-7a283716ed68.metadata.json

✅ Iceberg metadata.json fetched
  format-version : 2
  table-uuid     : 617a9854-1293-4181-b96c-c519fc4ba1c2
  location       : s3a://data-catalog-bucket/test-catalog/test_namespace/test_table

📐 Schema:
  [1] id : long
  [2] name : string
  [3] created_at : timestamp

📸 Snapshots:
  snapshot-id   : 2493930868147842358
  operation     : append
  added-records : 3


## Layer 3 — MinIO/data (Parquet files)

List all files in the table's data directory to verify Parquet files exist.

In [9]:
# Cell 4 — Layer 3: Verify Parquet files in MinIO
import boto3
from botocore.client import Config

print('=' * 60)
print('LAYER 3: MinIO/data (Parquet files)')
print('=' * 60)

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_URL,
    aws_access_key_id=MINIO_USER,
    aws_secret_access_key=MINIO_PASS,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# List all objects in table directory
prefix = f'{CATALOG}/{NAMESPACE}/{TABLE}/'
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)

files = []
for obj in response.get('Contents', []):
    key  = obj['Key']
    size = obj['Size']
    file_type = 'metadata' if '/metadata/' in key else 'data'
    files.append({'path': key, 'size_bytes': size, 'type': file_type})

df = pd.DataFrame(files)
print(f'\n📦 Total files: {len(files)}')

print('\n📁 Metadata files (Iceberg spec):')
meta_files = df[df['type'] == 'metadata']
display(meta_files[['path', 'size_bytes']])

print('\n📊 Data files (Parquet):')
data_files = df[df['type'] == 'data']
display(data_files[['path', 'size_bytes']])

total = data_files['size_bytes'].sum()
print(f'\n💾 Total data size: {total} bytes')
print(f'📊 Parquet files: {len(data_files)}')
print('\n✅ Layer 3 verified — Parquet files stored in MinIO')

LAYER 3: MinIO/data (Parquet files)

📦 Total files: 6

📁 Metadata files (Iceberg spec):


,path,size_bytes
1,test-catalog/test_namespace/test_table/metadat...,896
2,test-catalog/test_namespace/test_table/metadat...,897
3,test-catalog/test_namespace/test_table/metadat...,1962
4,test-catalog/test_namespace/test_table/metadat...,6779
5,test-catalog/test_namespace/test_table/metadat...,4459



📊 Data files (Parquet):


,path,size_bytes
0,test-catalog/test_namespace/test_table/data/00...,960



💾 Total data size: 960 bytes
📊 Parquet files: 1

✅ Layer 3 verified — Parquet files stored in MinIO


In [10]:
# Cell 5 — Full picture summary
print('=' * 60)
print('COMPLETE THREE-LAYER ARCHITECTURE SUMMARY')
print('=' * 60)

print("""
LAYER 1: PostgreSQL (Polaris Metastore)
  Purpose : Who can access what (RBAC + entity registry)
  Manages : Realm, Principal, PrincipalRole, Catalog,
            Namespace, Table, CatalogRole, Grants
  Access  : Via Polaris Management + Catalog REST API

      ↓ Catalog entity stores metadata location pointer

LAYER 2: MinIO/metadata (Iceberg spec)
  Purpose : How data is organized (table format)
  Manages : Schema, Snapshots, Manifests, Partition spec
  Files   : metadata.json, manifest-list.avro, manifest.avro
  Access  : Via Iceberg REST Catalog API

      ↓ Manifest files store Parquet file locations

LAYER 3: MinIO/data (Parquet format)
  Purpose : Actual data storage
  Manages : Column data, compression, row groups
  Files   : *.parquet
  Access  : Direct S3 read by query engines
""")

print('Key insight:')
print('  Polaris = catalog layer (WHERE things are, WHO can access)')
print('  Iceberg = format layer (HOW things are organized)')
print('  MinIO   = storage layer (actual BYTES)')

COMPLETE THREE-LAYER ARCHITECTURE SUMMARY

LAYER 1: PostgreSQL (Polaris Metastore)
  Purpose : Who can access what (RBAC + entity registry)
  Manages : Realm, Principal, PrincipalRole, Catalog,
            Namespace, Table, CatalogRole, Grants
  Access  : Via Polaris Management + Catalog REST API

      ↓ Catalog entity stores metadata location pointer

LAYER 2: MinIO/metadata (Iceberg spec)
  Purpose : How data is organized (table format)
  Manages : Schema, Snapshots, Manifests, Partition spec
  Files   : metadata.json, manifest-list.avro, manifest.avro
  Access  : Via Iceberg REST Catalog API

      ↓ Manifest files store Parquet file locations

LAYER 3: MinIO/data (Parquet format)
  Purpose : Actual data storage
  Manages : Column data, compression, row groups
  Files   : *.parquet
  Access  : Direct S3 read by query engines

Key insight:
  Polaris = catalog layer (WHERE things are, WHO can access)
  Iceberg = format layer (HOW things are organized)
  MinIO   = storage layer (actua

## 관찰 결과 (Observations)

- PostgreSQL은 엔티티 메타데이터만 저장 — 실제 데이터 없음
- Iceberg metadata.json은 스키마, 스냅샷, 파티션 정보 포함
- Parquet 파일은 실제 row 데이터만 포함
- 세 레이어가 pointer chain으로 연결됨

## 핵심 학습 포인트 (Key Takeaways)

1. Polaris/PostgreSQL은 **무엇이 어디있는지** 관리
2. Iceberg/MinIO metadata는 **어떻게 구성되어있는지** 관리  
3. Parquet/MinIO data는 **실제 데이터** 저장
4. 쿼리 엔진은 이 세 레이어를 순서대로 탐색하여 데이터 반환